# Storage: Volumes, PVs & PVCs

## What's covered

- **Why storage matters** — the container filesystem is ephemeral, and restarts wipe it
- **Volumes** — `spec.volumes` plus `volumeMounts`, the union model that ties storage to containers
- **Ephemeral volume types** — `emptyDir`, `hostPath`, `projected`; what each is for and what makes `hostPath` dangerous
- **The persistent storage abstraction** — `PersistentVolume` (PV), `PersistentVolumeClaim` (PVC), `StorageClass`, and why decoupling them is the whole point
- **Access modes & volume modes** — `RWO`, `ROX`, `RWX`, `RWOP`; `Filesystem` vs `Block`
- **`StorageClass`** — dynamic provisioning, default classes, provisioners
- **Reclaim policies** — `Retain`, `Delete`, and the deprecated `Recycle`
- **The Container Storage Interface (CSI)** — how Kubernetes talks to storage backends without knowing what they are
- **StatefulSets and `volumeClaimTemplates`** — the canonical way to give every pod its own PVC
- **Resizing PVCs** — when it works and when it doesn't

By the end of this notebook you should be able to attach scratch space to a pod, request real persistent storage with a single YAML file, understand exactly what binding means, and explain why every database in Kubernetes uses a StatefulSet plus `volumeClaimTemplates` instead of a Deployment. Notebooks 02 and 03 are the prerequisites — pod manifests and Deployments — because every storage manifest plugs into a pod template.

## Why storage matters — the container filesystem is ephemeral

When a container exits and the kubelet restarts it, the container's filesystem is **wiped**. Specifically:

- The image's read-only layers stay (they're cached on the node).
- The container's writable top layer is **deleted** when the container is removed.

That's by design — containers are supposed to be cattle, not pets. But it means anything the application wrote to `/var/lib/postgres`, `/data`, or any other "let me just save this here" path is gone the moment the container restarts. Including:

- Database files. Restarts wipe your data.
- Uploaded user files. Restarts wipe your uploads.
- Caches. Restarts wipe your caches (sometimes desirable, sometimes not).
- Even log files, if anything was reading them off disk.

You need storage that *outlives the container*. Kubernetes offers two levels.

**Pod-lifetime storage.** A volume that lives as long as the *pod* does. Shared between containers in the pod, deleted with the pod. Examples: `emptyDir`, `configMap`-as-volume, `secret`-as-volume. Good for scratch space, inter-process communication between sidecars, mounted config.

**Cluster-lifetime storage.** A volume backed by something external to the cluster (a cloud disk, an NFS server, a SAN LUN) that survives pod deletion, node failure, and rescheduling. This is what the `PersistentVolume` / `PersistentVolumeClaim` abstraction gives you.

For each workload, the question is "how long should the data live?" — and the answer picks the volume type.

## Volumes — the basics

A pod can declare a list of **volumes** under `spec.volumes`. Each container can mount any of those volumes under `volumeMounts`. The connection between the two is by *name*.

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: example
spec:
  containers:
    - name: app
      image: busybox
      volumeMounts:
        - name: scratch          # references the volume below
          mountPath: /workspace
  volumes:
    - name: scratch              # name; the type comes next
      emptyDir: {}               # the volume type
```

Two things to internalise:

**The volume is declared on the pod, mounted into the container.** That's how multiple containers share a volume — the sidecar pattern from notebook 02 — both containers declare a `volumeMount` referencing the same `volumes[].name`.

**A "volume type" is a single key under the volume entry.** `emptyDir: {}`, `hostPath: { path: /tmp }`, `configMap: { name: cfg }`, `secret: { secretName: s }`, `persistentVolumeClaim: { claimName: data }`. Each volume entry uses **exactly one** of these type keys.

The mount itself supports a few useful knobs:

- **`readOnly: true`** — mount read-only.
- **`subPath: <relative-path>`** — mount only a sub-path of the volume at the target. Met in notebook 05 — the "doesn't live-update" gotcha.
- **`mountPropagation`** — for advanced cases like CSI drivers and host filesystem sharing. Rarely set by hand.

The volume types fall into two big groups: **ephemeral** (live with the pod) and **persistent** (outlive the pod, backed by a PV). The rest of this notebook walks both.

## Ephemeral volume types — `emptyDir`, `hostPath`, `projected`

### `emptyDir` — scratch space shared in the pod

A brand-new empty directory created when the pod is scheduled and deleted when the pod is removed. **No persistence**. Restarts of the *container* keep it; rescheduling the *pod* (to a new node) wipes it.

```yaml
volumes:
  - name: scratch
    emptyDir: {}                 # default: backed by the node's disk
  - name: ramdisk
    emptyDir:
      medium: Memory             # tmpfs — RAM-backed, fast, counts against memory limits
      sizeLimit: 512Mi
```

Use it for:

- Inter-process communication between sidecars (the log-shipper pattern from notebook 02).
- Scratch space that's faster than `/tmp` and isolated per pod.
- Buffers for processes that genuinely don't need durability.

### `hostPath` — a path on the node, mounted into the pod

```yaml
volumes:
  - name: host-logs
    hostPath:
      path: /var/log
      type: Directory            # validate: File, Directory, Socket, etc.
```

**`hostPath` is a footgun.** It punches a hole through the pod's isolation, exposes node internals, makes the pod non-portable, and is a security anti-pattern. The legitimate uses are narrow:

- Node-level agents (the DaemonSet from notebook 03) that *need* to read host paths — log shippers reading `/var/log`, metrics agents reading `/proc`, CNI plugins writing `/etc/cni/net.d`.
- Cluster-bootstrap tooling.
- Local development on a single-node cluster where simplicity matters more than safety.

For everything else — including persistence — use a PV. `hostPath` ties the pod's data to one specific node; the next time the scheduler picks a different node, the data is gone.

### `projected` — combining multiple sources into one mount

```yaml
volumes:
  - name: combined
    projected:
      sources:
        - configMap:
            name: app-config
        - secret:
            name: db-credentials
        - downwardAPI:
            items:
              - path: labels
                fieldRef:
                  fieldPath: metadata.labels
```

A `projected` volume mounts several ConfigMaps, Secrets, and downward-API items into the **same directory**. Useful when an app expects all its config under one path (`/etc/app/`) but the data comes from different sources.

### Plus the ConfigMap / Secret / downwardAPI volumes you already met

From notebook 05. They behave like ephemeral volumes — they live with the pod, content materialised by the kubelet at mount time.

In [ ]:
# Two containers sharing an emptyDir — one writes, the other reads
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: shared-scratch
spec:
  restartPolicy: Never
  containers:
    - name: writer
      image: busybox:1.36
      command: ["sh", "-c"]
      args:
        - |
          for i in 1 2 3 4 5; do
            echo "$(date) message $i" >> /shared/log.txt
            sleep 1
          done
      volumeMounts:
        - name: scratch
          mountPath: /shared
    - name: reader
      image: busybox:1.36
      command: ["sh", "-c", "sleep 7; echo '--- reader sees:'; cat /shared/log.txt"]
      volumeMounts:
        - name: scratch
          mountPath: /shared
  volumes:
    - name: scratch
      emptyDir: {}
EOF
!sleep 14
!kubectl logs shared-scratch -c reader

## The persistent storage abstraction — PV, PVC, StorageClass

For storage that *outlives* the pod, Kubernetes splits the problem in two:

- The **storage admin** provisions storage and exposes it as a **`PersistentVolume`** (PV) — a cluster-scoped object that represents a piece of physical storage (a 20 GB cloud disk, an NFS share, a directory on a node).
- The **application author** asks for storage by writing a **`PersistentVolumeClaim`** (PVC) — a namespaced object that says "I want 5 GiB of `ReadWriteOnce` storage from the `fast-ssd` class."

A controller binds the two together. The pod references the PVC by name; the PVC is bound to a PV; the PV is the actual disk.

```
+-----------+   references    +-----+    bound to    +----+
|   Pod     | --------------> | PVC | -------------- | PV |   one piece of real storage
+-----------+                 +-----+                +----+
                              namespace-scoped       cluster-scoped
                              ("I want N GiB")       ("here's the disk")
```

The decoupling matters. The app says *what* it needs (size, access mode, class). The PV says *what's available* (size, access mode, backend). The binding controller matches them. Apps don't know whether they're getting an EBS volume, an NFS share, or a Ceph RBD — and they don't have to.

### Two ways the PV side gets created

**Static provisioning.** A cluster admin pre-creates `PersistentVolume` objects ahead of time. PVCs bind to whichever pre-existing PV best fits. Useful for hand-curated storage (a corp NFS server, on-prem SAN).

**Dynamic provisioning.** The admin creates a `StorageClass` describing *how* to make a disk; PVCs reference the class, and a controller (the **CSI external-provisioner** for that backend) creates a new PV on demand. The everyday case in cloud clusters.

For dynamic provisioning to work, the cluster needs a `StorageClass`. Most managed Kubernetes services ship one as the *default*. On `kind`, the default is a small provisioner called `local-path` that creates a directory on the node — fine for learning, not for production.

## Access modes and volume modes

A PVC says how the workload wants to *use* the volume. Two orthogonal axes.

### Access modes

| Mode | Code | Meaning |
|---|---|---|
| `ReadWriteOnce` | `RWO` | One *node* can mount it read-write. The default and most common. Pods on the same node can share it. |
| `ReadOnlyMany` | `ROX` | Many nodes can mount it read-only. Rare. |
| `ReadWriteMany` | `RWX` | Many nodes can mount it read-write. Requires a backend that supports it (NFS, CephFS, Azure Files). **Not** supported by cloud block storage (EBS, Persistent Disk). |
| `ReadWriteOncePod` | `RWOP` | Only one *pod* in the entire cluster can mount it. Stronger than `RWO`, available in Kubernetes 1.27+. |

A PVC requests one or more access modes; the bound PV must support at least one of them. For cloud block storage (EBS, GCE PD, Azure Disk) you get `RWO` by default — that's why a Deployment with one PVC won't easily scale to two replicas: the second replica scheduled on a different node can't mount the same `RWO` disk.

### Volume modes

| Mode | What you get |
|---|---|
| `Filesystem` *(default)* | The volume is formatted and mounted at the `mountPath`. Files and directories. |
| `Block` | The volume is exposed as a raw block device — the container sees a `/dev/xvda`-like device. Used by databases that manage their own on-disk format (some Cassandra setups, raw key/value stores). |

99% of workloads want `Filesystem`. `Block` mode is there when you have an app that wants raw devices.

## `StorageClass` — dynamic provisioning

A `StorageClass` is the recipe for "how do you make a new disk."

```yaml
apiVersion: storage.k8s.io/v1
kind: StorageClass
metadata:
  name: fast-ssd
  annotations:
    storageclass.kubernetes.io/is-default-class: "true"   # optional — mark as default
provisioner: ebs.csi.aws.com                              # the CSI driver
parameters:
  type: gp3
  iops: "6000"
  throughput: "250"
reclaimPolicy: Delete                                     # what happens when the PVC is deleted
volumeBindingMode: WaitForFirstConsumer                   # when to actually create the disk
allowVolumeExpansion: true                                # can the PVC be resized later?
```

The fields that matter day-to-day:

- **`provisioner`** — which CSI driver to use. The cluster has these installed; you reference them. Common ones: `ebs.csi.aws.com`, `pd.csi.storage.gke.io`, `disk.csi.azure.com`, `nfs.csi.k8s.io`, `rancher.io/local-path` (on `kind`).
- **`parameters`** — driver-specific. The EBS driver takes `type`, `iops`, `encrypted`; the NFS driver takes `server`, `share`.
- **`reclaimPolicy`** — covered below. Almost always `Delete` for dynamically-provisioned classes, `Retain` for hand-curated ones.
- **`volumeBindingMode`** — `Immediate` (default; pick a PV when the PVC is created) or `WaitForFirstConsumer` (wait until a pod schedules, then provision the disk in the pod's zone). Use `WaitForFirstConsumer` whenever the storage is zonal — otherwise you risk creating a disk in one zone and scheduling the pod in another.
- **`allowVolumeExpansion`** — set `true` if the driver supports online resize. Most cloud drivers do.

A PVC picks a class by name:

```yaml
spec:
  storageClassName: fast-ssd
  # ...
```

If you omit `storageClassName`, the cluster's *default* StorageClass is used. If there's no default and you omit the field, the PVC stays unbound until you add one.

### Listing what your cluster has

```bash
kubectl get storageclass                # what's installed
kubectl get sc -o wide                  # short alias
kubectl describe sc <name>              # parameters, provisioner, options
```

On `kind` you'll see `standard` (or `local-path`) marked default — that's the local-path provisioner shipping by default.

In [ ]:
# What StorageClasses does this cluster have?
!kubectl get storageclass
!echo '---'
# A PVC requesting 256 MiB of default-class storage
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: data
spec:
  accessModes:
    - ReadWriteOnce
  resources:
    requests:
      storage: 256Mi
EOF
!echo '---'
# kind's default class uses WaitForFirstConsumer — the PVC stays Pending until a pod uses it
!kubectl get pvc data
!echo '---'
# A pod that mounts the PVC and writes some data
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: pvc-writer
spec:
  restartPolicy: Never
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c"]
      args:
        - |
          echo "writing to /data..."
          date > /data/created-at
          echo "hello from pod" > /data/hello.txt
          ls -l /data
          sleep 30
      volumeMounts:
        - name: data
          mountPath: /data
  volumes:
    - name: data
      persistentVolumeClaim:
        claimName: data
EOF
!sleep 12
# Now the PVC is Bound and a PV exists for it
!kubectl get pvc data
!kubectl get pv | head -5
!echo '---'
!kubectl logs pvc-writer

## Reclaim policies — what happens when a PVC is deleted

When you delete a PVC, the bound PV's **reclaim policy** decides the fate of the underlying disk:

| Policy | Meaning |
|---|---|
| `Delete` | Delete the underlying disk. The PV object is removed too. **Data is gone.** The default for dynamically-provisioned PVs. |
| `Retain` | Keep the disk and the PV object. The PV becomes `Released` (not `Available`) — an admin has to clean it up before another PVC can bind. Use this for hand-curated, valuable storage. |
| `Recycle` | *Deprecated.* Used to wipe the volume and make it `Available` again. Don't use; use dynamic provisioning instead. |

Where the policy comes from:

- For **dynamically-provisioned** PVs, the policy is inherited from the StorageClass (typically `Delete`).
- For **statically-provisioned** PVs, you set the policy on the PV object directly.

You can switch a PV's policy with `kubectl patch`:

```bash
kubectl patch pv <pv-name> -p '{"spec":{"persistentVolumeReclaimPolicy":"Retain"}}'
```

Worth doing on the PV that holds your prod database, even if the StorageClass says `Delete` — defence in depth against an accidental PVC deletion.

### Released vs Available

When a PV is reclaimed with `Retain`, it goes to `Released` — it's not bound to a PVC but it still has the leftover data and a `claimRef` to the deleted PVC. To make it usable again, an admin removes the `claimRef`. Until then, no new PVC will bind to it.

## The Container Storage Interface — how Kubernetes talks to storage

Kubernetes doesn't know how to create a disk on AWS, or mount an NFS share, or attach a Ceph RBD. The work is done by **CSI drivers** — plugins installed in the cluster that implement a standard gRPC interface.

A CSI driver is two parts:

- A **controller** (Deployment) — talks to the storage backend's API. Creates and deletes volumes, takes snapshots, expands disks.
- A **node component** (DaemonSet) — runs on every node. Attaches volumes to the node and mounts them into the pod's filesystem.

The big idea: every storage backend speaks the same CSI gRPC, so Kubernetes itself stays storage-agnostic. Each backend (AWS, GCP, Azure, NFS, Ceph, Portworx, Longhorn, hostpath, local-path) ships its own driver.

You'll rarely write CSI code. What matters for the CKA:

- **Drivers are typically Helm-installed** at cluster setup time, and you reference them by *provisioner name* in your StorageClass.
- **`kubectl get csidrivers`** lists what's installed.
- **`kubectl get csinodes`** shows which drivers each node has registered.
- **Snapshots, clones, expansion** are all CSI-driven features. The driver decides which are supported.

### In-tree vs out-of-tree

Older Kubernetes shipped storage drivers *in-tree* (built into the kubelet). They've been deprecated — every cloud provider now ships a CSI driver instead. New clusters install drivers as separate pods. You may still encounter older `volume.beta.kubernetes.io/storage-provisioner` annotations on legacy clusters; treat them as the same idea, older mechanism.

## StatefulSets and `volumeClaimTemplates` — per-pod PVCs

You met StatefulSet briefly in notebook 03 — pods with stable identity (`pod-0`, `pod-1`). The storage part of that story is **`volumeClaimTemplates`**.

A Deployment shares its pod template across all replicas. If you put a `persistentVolumeClaim` reference in a Deployment's template, every replica mounts the *same* PVC — and that PVC can usually only be `RWO`-mounted on one node, so you can't scale past one replica.

A StatefulSet solves this. Instead of `spec.volumes` holding one PVC reference, you write a `volumeClaimTemplate`:

```yaml
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: cassandra
spec:
  serviceName: cassandra        # headless Service (notebook 04)
  replicas: 3
  selector:
    matchLabels:
      app: cassandra
  template:
    metadata:
      labels:
        app: cassandra
    spec:
      containers:
        - name: app
          image: cassandra:5
          volumeMounts:
            - name: data
              mountPath: /var/lib/cassandra
  volumeClaimTemplates:
    - metadata:
        name: data
      spec:
        accessModes: [ReadWriteOnce]
        resources:
          requests:
            storage: 10Gi
```

What the controller does:

1. For each replica `i` from 0 to `replicas - 1`, creates a PVC named `<template-name>-<sts-name>-<i>` — so `data-cassandra-0`, `data-cassandra-1`, `data-cassandra-2`.
2. Each PVC binds to its own PV. Each pod mounts its own PVC at `/var/lib/cassandra`.
3. If a pod is restarted or rescheduled, **the PVC stays**. The new `pod-0` mounts the same `data-cassandra-0` as before. Stable identity, stable storage.

### Deletion behaviour

Deleting the StatefulSet **does not** delete the PVCs. That's deliberate — your data is precious. To clean up the PVCs you delete them manually:

```bash
kubectl delete pvc -l app=cassandra
```

Kubernetes 1.27+ added `persistentVolumeClaimRetentionPolicy` on the StatefulSet spec to opt in to cascade deletion. Read the docs and choose deliberately.

### Why every database in Kubernetes uses this

Stable name (`cassandra-0`), stable network identity (`cassandra-0.cassandra.default.svc.cluster.local` via the headless Service), stable storage (`data-cassandra-0`). That triplet is exactly what a database replica needs.

In [ ]:
# A small StatefulSet (two replicas, to keep things light), with its headless Service
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Service
metadata:
  name: cache
spec:
  clusterIP: None
  selector:
    app: cache
  ports:
    - port: 80
      name: http
---
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: cache
spec:
  serviceName: cache
  replicas: 2
  selector:
    matchLabels:
      app: cache
  template:
    metadata:
      labels:
        app: cache
    spec:
      containers:
        - name: app
          image: busybox:1.36
          command: ["sh", "-c", "echo started; sleep 3600"]
          volumeMounts:
            - name: data
              mountPath: /data
  volumeClaimTemplates:
    - metadata:
        name: data
      spec:
        accessModes: [ReadWriteOnce]
        resources:
          requests:
            storage: 128Mi
EOF
!sleep 25
# Each replica got its own PVC, bound to its own PV
!kubectl get pods -l app=cache
!echo '---'
!kubectl get pvc -l app=cache
!echo '---'
# Write to pod 0; the data sticks to its PVC
!kubectl exec cache-0 -- sh -c 'echo "pod 0 was here" > /data/note.txt'
!kubectl exec cache-0 -- cat /data/note.txt
!echo '---'
# Pod 1 has its OWN /data — empty
!kubectl exec cache-1 -- ls /data

## Resizing PVCs

You requested 10 GiB; it's full; you want 20. If your StorageClass has `allowVolumeExpansion: true` and your CSI driver supports it, the steps are:

```bash
kubectl patch pvc data -p '{"spec":{"resources":{"requests":{"storage":"20Gi"}}}}'
# or
kubectl edit pvc data
```

The controller resizes the underlying disk, then the kubelet expands the filesystem inside the pod (online for most modern filesystems, no restart needed).

A few constraints:

- **You can only grow, not shrink.** Smaller is rejected by the API server.
- **Not every driver supports it.** `kind`'s `local-path` doesn't; cloud drivers usually do.
- **The PVC has a `FileSystemResizePending` condition** until the filesystem-side expansion completes. `kubectl describe pvc <name>` shows it.

If `allowVolumeExpansion: false` on the StorageClass, the API server rejects the patch. The fix is to edit the StorageClass (or use a different one) and recreate the PVC — which means data migration, not a one-liner.

## Cleaning up

Delete everything this notebook created so the cluster's tidy for notebook 07:

```bash
kubectl delete pod shared-scratch pvc-writer
kubectl delete statefulset cache
kubectl delete service cache
kubectl delete pvc data data-cache-0 data-cache-1
```

The PVs themselves go too — `local-path`'s reclaim policy is `Delete`, so deleting the PVCs cleans up the disks.

Or, more broadly:

```bash
kubectl delete pod,pvc,statefulset --all
kubectl delete service cache 2>/dev/null
```